# Socioeconomic Priority Analysis for Indonesian Provinces

This notebook aggregates district/city socioeconomic indicators by province and year, clusters provincial profiles, calculates need scores, and presents the results in charts and an interactive map.

## 1. Analysis Setup

Import the libraries used for data processing, clustering, evaluation, visualization, and mapping.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn_extra.cluster import KMedoids

import warnings
warnings.filterwarnings('ignore')


## 2. Load the Dataset

Load the dataset from the local `new_dataset.csv` file; no Kaggle download is required.

In [ ]:
dataset_path = Path('new_dataset.csv')
df = pd.read_csv(dataset_path)
print(f'Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()


## 3. Select Columns and Clean the Data

Select relevant indicators, normalize text, convert numeric data types, and handle duplicate rows and missing values.

In [ ]:
selected_columns = [
    'province', 'cities_reg', 'year',
    'poorpeople_percentage', 'reg_gdp', 'life_exp', 'avg_schooltime', 'exp_percap'
]

df = df[selected_columns].copy()
df.columns = [col.strip() for col in df.columns]

numeric_columns = ['poorpeople_percentage', 'reg_gdp', 'life_exp', 'avg_schooltime', 'exp_percap']
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['province'] = df['province'].astype(str).str.strip().str.upper()
df['cities_reg'] = df['cities_reg'].astype(str).str.strip().str.title()

df = df.drop_duplicates()
df = df.dropna(subset=['province', 'cities_reg', 'year', *numeric_columns])
df[numeric_columns] = df[numeric_columns].fillna(df[numeric_columns].median())
df['year'] = df['year'].astype(int)

df.head()


## 4. Data Quality Checks

Review missing values and summary statistics to confirm the data is ready for analysis.

In [ ]:
df.isnull().sum()


In [ ]:
df.describe(include=[np.number]).T


## 5. Aggregate Indicators by Province and Year

District/city values are summarized as average indicators for each province-year combination.

In [ ]:
feature_columns = ['poorpeople_percentage', 'reg_gdp', 'life_exp', 'avg_schooltime', 'exp_percap']

df_merged = (
    df.groupby(['province', 'year'], as_index=False)[feature_columns]
    .mean()
    .sort_values(['year', 'province'])
    .reset_index(drop=True)
)

df_merged.head()


## 6. Feature Scaling

Standardize features before clustering so differences in indicator scales do not dominate distance calculations.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_merged[feature_columns])
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_columns, index=df_merged.index)
X_scaled_df.head()


## 7. Clustering

Compare K-Means, Agglomerative Clustering, DBSCAN, Gaussian Mixture, and K-Medoids.

In [ ]:
n_clusters = 5

kmeans = KMeans(n_clusters=n_clusters, random_state=42)
df_merged['Cluster_KMeans'] = kmeans.fit_predict(X_scaled)

hc = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
df_merged['Cluster_HC'] = hc.fit_predict(X_scaled)

dbscan = DBSCAN(eps=2.0, min_samples=3)
df_merged['Cluster_DBSCAN'] = dbscan.fit_predict(X_scaled)

gmm = GaussianMixture(n_components=n_clusters, random_state=42)
df_merged['Cluster_GMM'] = gmm.fit_predict(X_scaled)

kmedoids = KMedoids(n_clusters=n_clusters, random_state=42)
df_merged['Cluster_KMedoids'] = kmedoids.fit_predict(X_scaled)
df_merged.head()


## 8. Model Evaluation

Evaluate clustering results using the Silhouette Score (higher is better) and Davies-Bouldin Index (lower is better).

In [ ]:
model_columns = {
    'KMeans': 'Cluster_KMeans',
    'HC': 'Cluster_HC',
    'DBSCAN': 'Cluster_DBSCAN',
    'GMM': 'Cluster_GMM',
    'KMedoids': 'Cluster_KMedoids'
}

evaluation_results = []

for model, col_name in model_columns.items():
    labels = df_merged[col_name]
    unique_labels = set(labels)
    if len(unique_labels - {-1}) > 1:
        sil_score = silhouette_score(X_scaled, labels)
        db_score = davies_bouldin_score(X_scaled, labels)
    else:
        sil_score = np.nan
        db_score = np.nan

    evaluation_results.append({
        'Model': model,
        'Silhouette Score (Higher is Better)': sil_score,
        'Davies-Bouldin Index (Lower is Better)': db_score
    })

eval_df = pd.DataFrame(evaluation_results)
eval_df.sort_values(by='Silhouette Score (Higher is Better)', ascending=False)


## 9. Need Score and Priority

Select the best model by Silhouette Score. Normalize indicators to a 0?1 range, align their directions, and summarize the result as a 0?100 need score.

In [ ]:
best_model_row = (
    eval_df.dropna(subset=['Silhouette Score (Higher is Better)'])
    .sort_values(by='Silhouette Score (Higher is Better)', ascending=False)
    .iloc[0]
)
best_model_name = best_model_row['Model']
best_cluster_col = model_columns[best_model_name]
sil_score_value = best_model_row['Silhouette Score (Higher is Better)']
print(f"Best clustering model: {best_model_name} (Silhouette Score = {sil_score_value:.3f})")

df_merged['Final_Cluster'] = df_merged[best_cluster_col]

score_features = {
    'poorpeople_percentage': 1,
    'reg_gdp': -1,
    'life_exp': -1,
    'avg_schooltime': -1,
    'exp_percap': -1
}

score_data = df_merged[list(score_features)].copy()
normalized = pd.DataFrame(
    MinMaxScaler().fit_transform(score_data),
    columns=score_data.columns,
    index=score_data.index
)

for column, direction in score_features.items():
    if direction == -1:
        normalized[column] = 1 - normalized[column]

priority_df = pd.DataFrame(index=df_merged.index)
priority_df['Need_Score'] = normalized.mean(axis=1) * 100
priority_df['Cluster'] = df_merged['Final_Cluster']

cluster_avg_score = priority_df.groupby('Cluster')['Need_Score'].mean().sort_values(ascending=False)
cluster_order = cluster_avg_score.index.tolist()
label_order = ['Very High', 'High', 'Medium', 'Low', 'Very Low']
label_map = {cluster_id: label_order[i] if i < len(label_order) else f'Cluster {cluster_id}' for i, cluster_id in enumerate(cluster_order)}
rank_map = {cluster_id: idx + 1 for idx, cluster_id in enumerate(cluster_order)}

priority_df['Priority'] = priority_df['Cluster'].map(label_map)
priority_df['Cluster_Rank'] = priority_df['Cluster'].map(rank_map)

priority_df = priority_df.join(df_merged[['province', 'year']])
priority_df.sort_values(['Cluster_Rank', 'Need_Score'], ascending=[True, False]).head(10)


## 10. Visual Summary

The chart below compares the average need score across priority groups.

In [ ]:
summary_cluster = (
    priority_df.groupby('Priority', as_index=False)['Need_Score']
    .mean()
    .sort_values('Need_Score', ascending=False)
)
print(summary_cluster)

plt.figure(figsize=(10, 5))
sns.barplot(data=summary_cluster, x='Priority', y='Need_Score', palette='viridis')
plt.title('Average Need Score by Cluster')
plt.ylabel('Need Score (0?100)')
plt.xlabel('Priority')
plt.tight_layout()
plt.show()


## 11. Interactive Priority Map

The map displays province scores for the latest available year. Darker red indicates a higher need score; hover over a province to view its indicators.

In [ ]:
import requests
import folium
from IPython.display import display

# Display the latest year available in the dataset.
map_year = int(priority_df['year'].max())
map_data = priority_df.loc[
    priority_df['year'].eq(map_year),
    ['province', 'year', 'Need_Score', 'Priority', 'Cluster']
].merge(
    df_merged[['province', 'year', *score_features]],
    on=['province', 'year'],
    how='left',
    validate='one_to_one'
)

# This GeoJSON contains boundaries for all 38 Indonesian provinces.
geojson_url = (
    'https://raw.githubusercontent.com/denyherianto/'
    'indonesia-geojson-topojson-maps-with-38-provinces/main/'
    'GeoJSON/indonesia-38-provinces.geojson'
)
response = requests.get(geojson_url, timeout=30)
response.raise_for_status()
geojson = response.json()
for feature in geojson['features']:
    feature['properties']['PROVINSI_KEY'] = (
        str(feature['properties']['PROVINSI']).strip().upper()
    )

province_name_fixes = {
    'D I YOGYAKARTA': 'DAERAH ISTIMEWA YOGYAKARTA',
    'KEP. BANGKA BELITUNG': 'KEPULAUAN BANGKA BELITUNG'
}
map_data['province_key'] = (
    map_data['province'].str.strip().str.upper().replace(province_name_fixes)
)
geojson_provinces = {
    str(feature['properties']['PROVINSI']).strip().upper()
    for feature in geojson['features']
}
missing_geometry = sorted(set(map_data['province_key']) - geojson_provinces)
if missing_geometry:
    raise ValueError(f'Map geometry not found for provinces: {missing_geometry}')

map_lookup = map_data.set_index('province_key')
for feature in geojson['features']:
    province_key = feature['properties']['PROVINSI_KEY']
    if province_key in map_lookup.index:
        row = map_lookup.loc[province_key]
        feature['properties'].update({
            'Need_Score': round(float(row['Need_Score']), 2),
            'Priority': row['Priority'],
            'Cluster': int(row['Cluster']),
            'Year': int(row['year']),
            'Poverty Rate (%)': round(float(row['poorpeople_percentage']), 2),
            'Regional GDP': round(float(row['reg_gdp']), 2),
            'Life Expectancy': round(float(row['life_exp']), 2),
            'Average Years of Schooling': round(float(row['avg_schooltime']), 2)
        })

priority_map = folium.Map(
    location=[-2.5, 118],
    zoom_start=5,
    tiles='CartoDB positron',
    control_scale=True
)
folium.Choropleth(
    geo_data=geojson,
    data=map_data,
    columns=['province_key', 'Need_Score'],
    key_on='feature.properties.PROVINSI_KEY',
    threshold_scale=[0, 20, 40, 60, 80, 100],
    fill_color='YlOrRd',
    fill_opacity=0.8,
    line_opacity=0.35,
    nan_fill_color='lightgray',
    legend_name=f'Need score {map_year} (0-100; darker red indicates higher priority)'
).add_to(priority_map)

folium.GeoJson(
    geojson,
    name='Province details',
    style_function=lambda feature: {
        'fillOpacity': 0,
        'color': '#555555',
        'weight': 0.5
    },
    highlight_function=lambda feature: {'weight': 2, 'color': '#222222'},
    tooltip=folium.GeoJsonTooltip(
        fields=[
            'PROVINSI', 'Year', 'Need_Score', 'Priority', 'Cluster',
            'Poverty Rate (%)', 'Regional GDP', 'Life Expectancy', 'Average Years of Schooling'
        ],
        aliases=[
            'Province:', 'Year:', 'Need score:', 'Priority:', 'Cluster:',
            'Poverty rate (%):', 'Regional GDP:', 'Life expectancy:', 'Average years of schooling:'
        ],
        localize=True
    )
).add_to(priority_map)
folium.LayerControl().add_to(priority_map)
display(priority_map)
